# Prerequisites

In [ ]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Defind the rdd
rdd = sc.textFile('/content/around_the_world_in_80_days.txt')

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

words is just an RDD object, not the actual data. flatMap is lazy so nothing runs yet.

Nothing is computed until we call an action like collect() or take().

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

collect() actually runs the transformations and brings back all the words as a normal python list, instead of just showing the RDD object.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
rdd.flatMap?

flatMap splits each line into words and puts them all in one flat list. map would give a list of lists instead (one list of words per line).

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [ ]:
# a. count the occurence of each word
word_counts = words.reduceByKey(lambda a, b: a + b)
word_counts.take(20)

In [ ]:
# b. a common first step in text analysis, change all capital letters to lower case
words_lower = rdd.flatMap(lambda l: l.split(' ')) \
                  .map(lambda w: (w.lower(), 1)) \
                  .reduceByKey(lambda a, b: a + b)
words_lower.take(20)

In [ ]:
# c. eliminate the stop words.
stopwords = ["a","the","is","in","it","of","and","to",
             "was","he","she","that","on","for","with",
             "as","at","by","an","be","this","his","her",
             "not","are","but","from","they","i"]

words_nostop = words_lower.filter(lambda x: x[0] not in stopwords)
words_nostop.take(20)

In [ ]:
# d. sort in alphabetical order
words_alpha = words_nostop.sortByKey()
words_alpha.take(20)

In [ ]:
# e. sort descending by word frequency
words_freq = words_nostop.sortBy(lambda x: x[1], ascending=False)
words_freq.take(20)

In [ ]:
# f. remove punctuations and blank spaces
import string
punctuation = string.punctuation

words_final = rdd.flatMap(lambda l: l.split(' ')) \
                  .map(lambda w: w.translate(str.maketrans('', '', punctuation))) \
                  .map(lambda w: (w.lower().strip(), 1)) \
                  .filter(lambda x: len(x[0]) > 0 and x[0] not in stopwords) \
                  .reduceByKey(lambda a, b: a + b)
words_final.take(20)

In [ ]:
# put all the steps together in one function
def clean_count(rdd, stopwords, punctuation=string.punctuation):
    return (rdd.flatMap(lambda l: l.split(' '))
               .map(lambda w: w.translate(str.maketrans('', '', punctuation)))
               .map(lambda w: (w.lower().strip(), 1))
               .filter(lambda x: len(x[0]) > 0 and x[0] not in stopwords)
               .reduceByKey(lambda a, b: a + b))

word_counts_rdd = clean_count(rdd, stopwords)
word_counts_rdd.sortBy(lambda x: x[1], ascending=False).take(20)

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # pair each age with a count of 1
  .map(lambda x: (x[0], (x[1], 1)))
  # sum ages and counts for each name
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # divide total age by count to get the average
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [ ]:
import time

start = time.time()
res1 = (rdd.flatMap(lambda l: l.split(' '))
           .map(lambda w: (w.lower(), 1))
           .filter(lambda x: x[0] not in stopwords)
           .reduceByKey(lambda a, b: a + b))
res1.count()
print("filter before reduceByKey:", time.time() - start)

start = time.time()
res2 = (rdd.flatMap(lambda l: l.split(' '))
           .map(lambda w: (w.lower(), 1))
           .reduceByKey(lambda a, b: a + b)
           .filter(lambda x: x[0] not in stopwords))
res2.count()
print("filter after reduceByKey:", time.time() - start)

Filtering before reduceByKey is faster because less data has to be shuffled.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [ ]:
!wget -nc -O le_tour_du_monde.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8
rdd_fr = sc.textFile('/content/le_tour_du_monde.txt')

stopwords_fr = ["le","la","les","de","des","du","un","une",
                "et","à","en","que","qui","dans","pour",
                "pas","sur","se","ne","ce","il","elle",
                "je","tu","nous","vous","ils","au","aux"]

word_counts_fr = clean_count(rdd_fr, stopwords_fr)

print("english unique words:", word_counts_rdd.count())
print("french unique words:", word_counts_fr.count())

word_counts_fr.sortBy(lambda x: x[1], ascending=False).take(10)

The french and english versions have different top words but a similar shape overall (names and common verbs come up a lot in both).